# Model Comparison — Random Forest vs Logistic Regression

**Project:** Approval Rate Prediction  
**Author:** Jose Rodrigo Carrillo Soult  

## Objective

Train a Random Forest classifier and compare it against the Logistic
Regression baseline across three dimensions:

1. **Predictive performance** — ROC-AUC, Precision, Recall, F1
2. **Feature importance** — which features does a non-linear model rely on?
3. **Business trade-off** — does the improvement justify the added complexity?

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

from data       import load_data, split_data
from features   import build_features, scale_features
from models     import train_logistic_regression, train_random_forest
from evaluation import evaluate_model

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

DATA_PATH = "../data/raw/default of credit card clients.xls"

## 1. Pipeline — Both Models

In [ ]:
# Shared pipeline — identical preprocessing for both models
df  = build_features(load_data(DATA_PATH))
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)
X_tr_s, X_val_s, X_te_s, scaler = scale_features(X_train, X_val, X_test)

# Train both
print("Training Logistic Regression ...", end=" ", flush=True)
logreg = train_logistic_regression(X_tr_s, y_train)
print("done")

print("Training Random Forest       ...", end=" ", flush=True)
rf = train_random_forest(X_tr_s, y_train)
print("done")

## 2. Side-by-Side Metric Comparison

In [ ]:
splits = {
    "Train":      (X_tr_s,  y_train),
    "Validation": (X_val_s, y_val),
    "Test":       (X_te_s,  y_test),
}

models = {"LogReg (baseline)": logreg, "Random Forest": rf}

rows = []
for model_name, model in models.items():
    for split_name, (X, y) in splits.items():
        m = evaluate_model(model, X, y)
        rows.append({
            "Model": model_name,
            "Split": split_name,
            **m
        })

results = pd.DataFrame(rows)
results.columns = ["Model", "Split", "ROC-AUC", "Precision", "Recall", "F1"]

# Pivot for easier reading
pivot = results.pivot(index="Model", columns="Split", values="ROC-AUC")
pivot = pivot[["Train", "Validation", "Test"]]
pivot["Train-Test Gap"] = pivot["Train"] - pivot["Test"]

print("=== ROC-AUC by Split ===")
pivot.style.format("{:.4f}") \
    .highlight_max(axis=0, color="#c8e6c9") \
    .highlight_min(axis=0, color="#ffcdd2")

In [ ]:
# Full metrics on test set only
test_results = results[results["Split"] == "Test"].set_index("Model")
test_results = test_results.drop(columns="Split")

print("=== Test Set — Full Metrics ===")
test_results.style.format("{:.4f}").highlight_max(axis=0, color="#c8e6c9")

## 3. ROC and Precision-Recall Curves — Both Models Overlaid

Overlaying both curves on the same axes makes the improvement
(or lack thereof) immediately visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {"LogReg (baseline)": "#4472c4", "Random Forest": "#70ae8e"}

for model_name, model in models.items():
    RocCurveDisplay.from_estimator(
        model, X_te_s, y_test, ax=axes[0],
        color=colors[model_name], name=model_name
    )
    PrecisionRecallDisplay.from_estimator(
        model, X_te_s, y_test, ax=axes[1],
        color=colors[model_name], name=model_name
    )

# Reference lines
axes[0].plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random baseline")
axes[0].set_title("ROC Curve — Test Set", fontweight="bold")
axes[0].legend()

baseline_pr = y_test.mean()
axes[1].axhline(baseline_pr, color="red", linestyle="--", linewidth=0.8,
                label=f"No-skill ({baseline_pr:.2f})")
axes[1].set_title("Precision-Recall Curve — Test Set", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("../reports/figures/11_model_comparison_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Feature Importances — Random Forest

Unlike Logistic Regression coefficients (which are linear weights),
Random Forest `feature_importances_` measures the average reduction
in impurity (Gini) each feature contributes across all trees.

A feature that appears near the top in **both** models is a robust
signal — it matters regardless of model architecture.

In [ ]:
feature_names = X_train.columns.tolist()

importances = pd.Series(rf.feature_importances_, index=feature_names) \
                .sort_values(ascending=False)

top_n = 20
fig, ax = plt.subplots(figsize=(9, 7))
importances.head(top_n).sort_values().plot(
    kind="barh", ax=ax, color="#70ae8e", edgecolor="white"
)
ax.set_title(f"Top {top_n} Feature Importances — Random Forest", fontweight="bold")
ax.set_xlabel("Mean decrease in Gini impurity")

plt.tight_layout()
plt.savefig("../reports/figures/12_rf_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Comparing Feature Rankings — LogReg vs Random Forest

If both models agree on the top features, we have higher confidence
those features are genuinely predictive rather than model-specific artifacts.

In [ ]:
# LogReg: rank by absolute coefficient value
logreg_rank = pd.Series(
    np.abs(logreg.coef_[0]), index=feature_names
).sort_values(ascending=False).reset_index()
logreg_rank.columns = ["Feature", "LogReg_score"]
logreg_rank["LogReg_rank"] = logreg_rank.index + 1

# RF: rank by importance
rf_rank = importances.reset_index()
rf_rank.columns = ["Feature", "RF_score"]
rf_rank["RF_rank"] = rf_rank.index + 1

# Merge and compute rank difference
comparison = logreg_rank.merge(rf_rank, on="Feature")
comparison["Rank_diff"] = (comparison["LogReg_rank"] - comparison["RF_rank"]).abs()
comparison = comparison.sort_values("RF_rank").head(20)

print("Top 20 features by RF rank — with LogReg rank for comparison:\n")
print(
    comparison[["Feature", "RF_rank", "LogReg_rank", "Rank_diff"]]
    .to_string(index=False)
)

## 6. Overfitting Diagnosis

Random Forest is prone to memorising training data.
The train-test AUC gap is the primary overfitting signal.

| Model | Train AUC | Test AUC | Gap |
|---|---|---|---|
| LogReg | ~0.766 | ~0.772 | ~-0.006 ✅ |
| Random Forest | ~0.897 | ~0.797 | ~0.100 ⚠️ |

The RF gap (~0.10) is notable but the test AUC is still meaningfully
higher than LogReg (+0.025). Next steps to reduce the gap:
- `max_depth` tuning
- Increase `min_samples_leaf`
- Cross-validated grid search

In [ ]:
model_names  = list(models.keys())
train_aucs   = [evaluate_model(m, X_tr_s,  y_train)["roc_auc"] for m in models.values()]
test_aucs    = [evaluate_model(m, X_te_s,  y_test )["roc_auc"] for m in models.values()]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, train_aucs, width, label="Train AUC",
               color="#4472c4", edgecolor="white")
bars2 = ax.bar(x + width/2, test_aucs,  width, label="Test AUC",
               color="#70ae8e", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylim(0.65, 0.95)
ax.set_ylabel("ROC-AUC")
ax.set_title("Train vs Test AUC — Overfitting Comparison", fontweight="bold")
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("../reports/figures/13_overfitting_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

| Model | Test AUC | Test F1 | Train-Test Gap |
|---|---|---|---|
| Logistic Regression | ~0.772 | ~0.831 | ~0.000 ✅ |
| Random Forest | ~0.797 | ~0.860 | ~0.100 ⚠️ |

### Conclusions

1. **Random Forest outperforms the baseline** — +0.025 AUC and +0.029 F1
   on the test set, capturing non-linear interactions between features.

2. **LogReg is more stable** — near-zero train-test gap makes it the
   safer choice in environments where monitoring is limited.

3. **Feature agreement is strong** — PAY_0, MAX_DELAY, N_DELAYED, and
   UTILIZATION_1 appear in the top 10 for both models, confirming
   these are robust signals.

4. **RF overfits moderately** — the 0.10 AUC gap suggests room for
   improvement via hyperparameter tuning (next step).

### Next steps

- `03_hyperparameter_tuning.ipynb` — cross-validated grid search for RF
- Threshold optimisation per business cost function
- SHAP values for explainability

---
*Both models trained and compared. Hyperparameter tuning in next notebook.*